# IoTパート　PiNodeセットアップ

このNotebookは，Raspberry Pi（Raspi）を用いて，IoTシステム（PiNode）を構築するためのセットアップ手順をまとめたものです．  
PiNode3のハードウェアと01_raspi_setup.ipynbの実施を前提に，以下の流れで進めます．
1. IoTシステム（PiNode）のセットアップ
2. データ収集と収集確認
3. データ自動送信  

実行はRaspiにアクセスしたPCのターミナルから行います．

## 1. IoTシステム（PiNode）のセットアップ

### 1.1 リポジトリのクローン
以下のコマンドを実行して，リポジトリのクローンを行います．
```bash
$ git clone [リポジトリURL]
```

リポジトリURLは以下
#### [PiNode3](https://github.com/MinenoLab/PiNode3.git)

＜インストールする前に設定＞  
・ラズパイのユーザ名：pinode3  
・ラズパイのホスト名：固定IPアドレスの下2桁を付ける（例：PiNode1-11）  

### 1.2-a ソフトウェアのインストール（リモートDBに自動送信する場合）
クローンしたリポジトリの中にある`install_remotedb.sh`を実行して，必要なソフトウェアのインストールを行います．
```bash
$ cd PiNode3
$ bash install_remotedb.sh
```
＜インストールする時に＞  
・InfluxDBのトークンの入力：APIキーを確認して入力（例：0123456789abcdef0123456789abcdef01）  
・接続先ホストIPアドレスの入力：アップロード先のホストIPアドレスを入力（例：192.168.XX.XX）  
・接続先ユーザ名の入力：アップロード先のユーザ名を入力

### 1.2-b ソフトウェアのインストール（ローカルDBに保存する場合）
クローンしたリポジトリの中にある`install.sh`を実行して，必要なソフトウェアのインストールを行います．
```bash
$ cd PiNode3
$ bash install.sh
```


## 2. データ収集と収集確認

データ収集は，サービスファイルが実行され，Raspiの起動と同時に開始されるようになっています．
データ収集が正常に行われているかは，以下のコマンドで確認できます．
1. 収集データが保存されているか確認する
```bash
$ ls -l /home/data/image/image*
$ ls -l /home/data/sensor/
```
2. サービスの状態を確認する
```bash
$ sudo systemctl status data_collector.service
$ journalctl -u data_collector.service -e
```

カメラが認識されていない場合は，以下のコマンドでカメラが認識されているか確認してください．
```bash
$ ls　/dev/ttyUSB*
```
カメラ名のルールはPiNode3/driver/usb/以下のファイルに記載されています．

## 3. データ自動送信

収集データは，Raspiのローカルストレージに保存されますが，これを自動的にPCに送信するための設定が行われています．  
送信は，データ収集同様に，サービスファイルが実行され，Raspiの起動と同時に開始されるようになっています．  

サービスファイルは全て`/etc/systemd/system/`に保存されており，`daily_rsync.service`がデータ送信のサービスファイルになります．  
データ送信間隔を変更したい場合は以下のコマンドでサービスファイルを編集してください．
```bash
$ sudo nano /etc/systemd/system/daily_rsync.timer
# 5分ごとに送信する場合は以下に変更
[Unit]
Description=Run upload every 5min
[Timer]
OnCalendar=*:0/5 #5分ごとに実行
RandomizedDelaySec=180 #実行時間を0~180秒ランダムに遅らせる（他の機器のデータ送信との混雑を避ける）
```

サービスファイルの編集後は，以下のコマンドでサービスを再起動してください．
```bash
$ sudo systemctl daemon-reload
$ sudo systemctl restart daily_rsync.service
```